# SkinCare — Kaggle GPU Eğitimi

ConvNeXt-Tiny, 4 sınıf (Acne, Eczema, Eye_Bags, Wrinkles), 384x384.

## Çalıştırmadan önce (sağ paneldeki Settings)

| Ayar | Değer |
|---|---|
| **Accelerator** | GPU P100 (veya T4 x2) |
| **Internet** | **On** — timm ağırlıkları ve git clone için zorunlu |
| **Input** | `skincare-orchestration-data` datasetini ekle |

Internet kapalıysa hiçbir şey çalışmaz: ne repo klonlanır ne de ImageNet
ön-eğitimli ağırlıklar iner. Kaggle bunun için telefon doğrulaması ister.

Beklenen süre: epoch başına ~15 sn, erken durmayla toplam **6-12 dakika**.

## 1. Ortam kontrolü

In [ ]:
import subprocess, torch, os

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "Accelerator kapali. Settings > Accelerator > GPU sec."

# Dataset mount noktasini bul (dataset adini degistirdiysen otomatik bulunur).
DATA_ROOT = None
for entry in sorted(os.listdir("/kaggle/input")):
    candidate = os.path.join("/kaggle/input", entry)
    for sub in ("orchestration_data", ""):
        path = os.path.join(candidate, sub) if sub else candidate
        if os.path.isdir(os.path.join(path, "train")):
            DATA_ROOT = path
            break
    if DATA_ROOT:
        break

assert DATA_ROOT, ("Veri bulunamadi. Sag panelden 'Add Input' ile "
                   "skincare-orchestration-data datasetini ekle.")
print(f"veri: {DATA_ROOT}")
print("siniflar:", sorted(os.listdir(os.path.join(DATA_ROOT, "train"))))

## 2. Kod ve bağımlılıklar

Eğitim kodu repodan klonlanır, böylece notebook ile repo hep aynı kalır.

In [ ]:
!pip install -q "timm>=0.9.0" "coremltools>=7.0" 2>&1 | tail -2
!git clone -q --depth 1 https://github.com/keremoztopuz/skincare_detection.git /kaggle/working/repo

import timm
print("timm", timm.__version__)
!ls /kaggle/working/repo/src

## 3. Kaggle yollarına uyarlama

`/kaggle/input` salt-okunur, bu yüzden veri oradan okunur ama tüm çıktılar
`/kaggle/working` altına yazılır. Diğer hiçbir hiperparametreye dokunulmuyor —
repodaki reçete aynen koşuyor.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/repo/src")

import config

config.DATA_DIR = DATA_ROOT
config.TRAIN_DIR = os.path.join(DATA_ROOT, "train")
config.VAL_DIR = os.path.join(DATA_ROOT, "val")
config.TEST_DIR = os.path.join(DATA_ROOT, "test")

OUT = "/kaggle/working/outputs"
config.CHECKPOINT_DIR = f"{OUT}/checkpoints"
config.MODEL_SAVE_PATH = f"{OUT}/model/best_model.pth"
config.THRESHOLDS_SAVE_PATH = f"{OUT}/model/thresholds.json"
config.TOP1_MODEL_SAVE_PATH = f"{OUT}/model/best_top1_model.pth"
config.TOP1_THRESHOLDS_SAVE_PATH = f"{OUT}/model/top1_thresholds.json"
config.LOGS_DIR = f"{OUT}/logs"
config.IMAGES_DIR = f"{OUT}/images"
for directory in (config.CHECKPOINT_DIR, f"{OUT}/model", config.LOGS_DIR, config.IMAGES_DIR):
    os.makedirs(directory, exist_ok=True)

# P100/T4 16 GB, 384px batch 16'yi rahat tasir. 32 denemek istersen burada degistir.
# config.BATCH_SIZE = 32

print(f"device={config.DEVICE} | batch={config.BATCH_SIZE} | lr={config.LEARNING_RATE}")
print(f"epochs<={config.NUM_EPOCHS} | patience={config.PATIENCE} | siniflar={config.CLASS_NAMES}")

## 4. Veri doğrulama

Eğitime başlamadan önce split sayıları ve sızıntı denetimi. `TEMIZ` görmelisin —
aksi halde metrikler iyimser çıkar.

In [ ]:
from dataset import load_data, audit_split_leakage, calculate_pos_weights

for split_name, split_dir in (("train", config.TRAIN_DIR),
                              ("val", config.VAL_DIR),
                              ("test", config.TEST_DIR)):
    paths, labels = load_data(split_dir)
    per_class = {name: labels.count(index) for index, name in enumerate(config.CLASS_NAMES)}
    print(f"{split_name:<6} {len(paths):>5} goruntu  {per_class}")

leaks = audit_split_leakage()
print("\nsizinti denetimi:", "TEMIZ" if not leaks else f"{len(leaks)} SIZINTI VAR")

## 5. Eğitim

İlk 5 epoch yalnızca sınıflandırma kafası, sonra son ConvNeXt aşaması açılır.
AMP CUDA'da otomatik devreye girer. En iyi model val AUROC ile seçilir,
8 epoch iyileşme olmazsa durur.

In [ ]:
import random, time
import numpy as np
from model import build_model, freeze_backbone, get_model_info
from train import train_model

torch.manual_seed(config.SEED)
random.seed(config.SEED)
np.random.seed(config.SEED)

model = freeze_backbone(build_model()).to(config.DEVICE)
print(get_model_info(model))

started = time.time()
train_model(model)
print(f"\ntoplam egitim suresi: {(time.time() - started) / 60:.1f} dakika")

## 6. Eğitim eğrisi

In [ ]:
import csv
import matplotlib.pyplot as plt

log_path = os.path.join(config.LOGS_DIR, "metrics_history.csv")
with open(log_path) as log_file:
    rows = list(csv.DictReader(log_file))

epochs = [int(r["epoch"]) for r in rows]
figure, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs, [float(r["train_loss"]) for r in rows], label="train")
axes[0].plot(epochs, [float(r["val_loss"]) for r in rows], label="val")
axes[0].set_title("loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(epochs, [float(r["auroc"]) for r in rows], label="AUROC")
axes[1].plot(epochs, [float(r["top1_accuracy"]) for r in rows], label="Top-1")
axes[1].plot(epochs, [float(r["f1"]) for r in rows], label="F1")
axes[1].set_title("metrikler"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

best = max(rows, key=lambda r: float(r["auroc"]))
print(f"en iyi epoch {best['epoch']}: AUROC={best['auroc']} F1={best['f1']} Top1={best['top1_accuracy']}")

## 7. Kalibrasyon

Sınıf başına sigmoid eşikleri ve **temperature** değeri ölçülür. Bu temperature,
iOS uygulamasındaki elle seçilmiş `1.8` sabitinin yerine geçecek.

In [ ]:
import calibrate as calibrate_module

calibrate_module.TEMPERATURE_SAVE_PATH = f"{OUT}/model/temperature.json"
thresholds, calibration_metrics = calibrate_module.calibrate_model(
    model_path=config.MODEL_SAVE_PATH,
    thresholds_path=config.THRESHOLDS_SAVE_PATH,
)

## 8. Test değerlendirmesi (TTA ile)

Test seti eğitimde hiç görülmedi ve sızıntı denetiminden temiz geçti,
yani bu sayılar dürüst.

In [ ]:
from evaluate import evaluate_model, print_results

metrics, all_labels, all_predictions = evaluate_model(
    save_path=config.MODEL_SAVE_PATH,
    thresholds_path=config.THRESHOLDS_SAVE_PATH,
    tta=True,
)
print_results(metrics, all_labels, all_predictions)

## 9. CoreML export (FP16)

Linux'ta dönüştürme çalışır; logit doğrulaması `predict` macOS gerektirdiği için
atlanır — indirdikten sonra Mac'te `python export/export.py --model-path ...`
ile doğrulayabilirsin.

In [ ]:
sys.path.insert(0, "/kaggle/working/repo/export")
from export import export_to_coreml

package_path = export_to_coreml(
    model_path=config.MODEL_SAVE_PATH,
    output_path=f"{OUT}/coreml/skin_disease.mlpackage",
)
!du -sh {package_path}

## 10. Çıktıları indir

Tek zip: ağırlıklar, eşikler, temperature, metrik geçmişi, grafikler ve
CoreML paketi. Sağdaki **Output** sekmesinden indir.

In [ ]:
import shutil

archive = shutil.make_archive("/kaggle/working/skincare_outputs", "zip", OUT)
print(f"{archive}  ({os.path.getsize(archive) / 1e6:.0f} MB)")

for root, _, files in os.walk(OUT):
    for file_name in sorted(files):
        full = os.path.join(root, file_name)
        print(f"  {os.path.relpath(full, OUT):<45} {os.path.getsize(full) / 1e6:>7.1f} MB")